<a href="https://colab.research.google.com/github/SikandarHussain6858/BigDataAnalyticslabs/blob/main/BDA_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Muhammad Sikandar Hussain 502808

## CS-404: Big Data Analytics

## Step 1: Load and inspect

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/Lab02_Student_Service_Data.csv')
sample = df.head(12).copy()
print(sample.head())
print('Shape:', sample.shape)


  student_id  age  study_hours  cgpa       city program  service_visits  \
0   S2026001   18         23.0  2.32  Islamabad    BSDS               0   
1   S2026002   26         15.5  2.61  Islamabad    BSCS               2   
2   S2026003   25         21.7  2.42  Islamabad    BSDS               2   
3   S2026004   22         12.7  2.85    Karachi    BSCS               1   
4   S2026005   22         15.1  1.93    unknown    BSDS               0   

  registered  
0         no  
1        yes  
2         no  
3         no  
4        yes  
Shape: (12, 8)


## Step 2: Record the Data Identity

In [6]:
print(sample.dtypes)
print(sample.isna().sum())
print('Duplicates:', sample.duplicated().sum())
print(sample['city'].value_counts(dropna=False))


student_id         object
age                 int64
study_hours       float64
cgpa              float64
city               object
program            object
service_visits      int64
registered         object
dtype: object
student_id        0
age               0
study_hours       1
cgpa              1
city              0
program           0
service_visits    0
registered        0
dtype: int64
Duplicates: 0
city
Islamabad    6
Karachi      3
unknown      1
Peshawar     1
Lahore       1
Name: count, dtype: int64


## Step 3: Apply Simple Filters

In [7]:
clean = sample.drop_duplicates().copy()
clean['city'] = clean['city'].replace('unknown', np.nan)
clean.loc[~clean['study_hours'].between(0, 60), 'study_hours'] = np.nan
clean.loc[~clean['cgpa'].between(0, 4), 'cgpa'] = np.nan
print(clean.isna().sum())


student_id        0
age               0
study_hours       1
cgpa              1
city              1
program           0
service_visits    0
registered        0
dtype: int64


## Step 4: Check What Changed

In [8]:
print('Rows before:', len(sample))
print('Rows after :', len(clean))
print(clean[['study_hours','cgpa']].describe())


Rows before: 12
Rows after : 12
       study_hours       cgpa
count    11.000000  11.000000
mean     17.000000   2.783636
std       4.626662   0.800353
min       8.400000   1.930000
25%      14.550000   2.245000
50%      15.500000   2.420000
75%      21.650000   3.380000
max      23.000000   4.000000


## Task 1 Explain and Identify the Raw Dataset

In [9]:
students = pd.read_csv('/content/Lab02_Student_Service_Data.csv')
print(students.shape)
print(students.isna().sum())
print('Duplicates:', students.duplicated().sum())
print(students['city'].value_counts(dropna=False))


(164, 8)
student_id        0
age               0
study_hours       6
cgpa              5
city              4
program           0
service_visits    0
registered        0
dtype: int64
Duplicates: 4
city
Islamabad     45
Karachi       37
Lahore        34
Rawalpindi    26
Peshawar      12
unknown        6
NaN            4
Name: count, dtype: int64


## Task 2: Analyze and Apply Suitable Filters

### 1. Remove duplicates and handle hidden city labels

In [10]:
clean_students = students.drop_duplicates().copy()

# Convert hidden "unknown" labels into actual missing values
clean_students['city'] = clean_students['city'].replace('unknown', np.nan)

print("Rows before:", len(students))
print("Rows after :", len(clean_students))

print("\nMissing values after converting 'unknown':")
print(clean_students.isna().sum())

Rows before: 164
Rows after : 160

Missing values after converting 'unknown':
student_id         0
age                0
study_hours        6
cgpa               5
city              10
program            0
service_visits     0
registered         0
dtype: int64


### 2. Apply range checks

In [11]:
# Invalid study hours → NaN
clean_students.loc[
    ~clean_students['study_hours'].between(0, 60),
    'study_hours'
] = np.nan

# Invalid CGPA → NaN
clean_students.loc[
    ~clean_students['cgpa'].between(0, 4),
    'cgpa'
] = np.nan

print("After range checks:")
print(clean_students[['study_hours', 'cgpa']].describe())

After range checks:
       study_hours        cgpa
count   151.000000  153.000000
mean     17.412583    2.931176
std       7.724932    0.550729
min       0.800000    1.490000
25%      11.450000    2.590000
50%      16.800000    2.980000
75%      21.600000    3.300000
max      41.300000    4.000000


### 3. Calculate IQR for study_hours

In [12]:
Q1 = clean_students['study_hours'].quantile(0.25)
Q3 = clean_students['study_hours'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

Q1: 11.45
Q3: 21.6
IQR: 10.150000000000002
Lower Limit: -3.775000000000004
Upper Limit: 36.825


## Task 3 — Preprocessing Pipeline

### 1. Create X and y

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Input features
X = clean_students.drop(columns=['student_id', 'registered'])

# Target
y = clean_students['registered'].map({'yes': 1, 'no': 0})

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nInput features:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

X shape: (160, 6)
y shape: (160,)

Input features:
['age', 'study_hours', 'cgpa', 'city', 'program', 'service_visits']

Target distribution:
registered
0    94
1    66
Name: count, dtype: int64


### 2. 75/25 Stratified Split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training data: (120, 6)
Testing data : (40, 6)

Training target distribution:
registered
0    70
1    50
Name: count, dtype: int64

Testing target distribution:
registered
0    24
1    16
Name: count, dtype: int64


### 3. Identify Numerical and Categorical Features

In [15]:
numeric_features = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = X.select_dtypes(
    include=['object']
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['age', 'study_hours', 'cgpa', 'service_visits']
Categorical features: ['city', 'program']


### 4. Numerical Preprocessing

In [16]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

### 5. Categorical Preprocessing

In [17]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

### 6. Combine Both Pipelines

In [18]:
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

### 7. Create Logistic Regression Pipeline and Classification Report

In [19]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Classification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    target_names=['Not Registered', 'Registered']
))

Classification Report:

                precision    recall  f1-score   support

Not Registered       0.62      0.75      0.68        24
    Registered       0.45      0.31      0.37        16

      accuracy                           0.57        40
     macro avg       0.54      0.53      0.52        40
  weighted avg       0.55      0.57      0.56        40



## Task 4 — Compare and Evaluate the Results

In [20]:
before_after = pd.DataFrame({
    'Check': [
        'Rows',
        'Duplicate rows',
        'Hidden unknown labels',
        'Missing values to handle'
    ],
    'Before preprocessing': [
        len(students),
        students.duplicated().sum(),
        (students['city'] == 'unknown').sum(),
        students.isna().sum().sum()
    ],
    'After preprocessing': [
        len(clean_students),
        clean_students.duplicated().sum(),
        (clean_students['city'] == 'unknown').sum(),
        clean_students.isna().sum().sum()
    ]
})

before_after

,Check,Before preprocessing,After preprocessing
0,Rows,164,160
1,Duplicate rows,4,0
2,Hidden unknown labels,6,0
3,Missing values to handle,15,26


### Chunk-Based Reading

In [21]:
counts = {'yes': 0, 'no': 0}

for chunk in pd.read_csv(
    '/content/Lab02_Student_Service_Data.csv',
    chunksize=50,
    usecols=['student_id', 'program', 'registered']
):

    c = chunk['registered'].value_counts()

    counts['yes'] += int(c.get('yes', 0))
    counts['no'] += int(c.get('no', 0))

print(counts)

{'yes': 68, 'no': 96}
